# 03 - Cotizaciones Avanzadas

Este notebook prueba las funcionalidades avanzadas de cotizaciones de la API de IOL.

## Funcionalidades:
- Cotizaciones masivas por tipo de instrumento
- Paneles de cotizaciones (MERVAL, etc.)
- Cotizacion detallada de titulos individuales
- Analisis de puntas y libro de ordenes

**Nota:** Requiere credenciales validas de IOL en el archivo `.env`

## Configuracion Inicial

In [ ]:
import sys
import os
from datetime import datetime
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath('../..'))

from pyIol import (
    IOLClient, IOLAPIError,
    CotizacionesMasivas, TituloCotizacion, PuntasCotizacion, CotizacionDetallada
)
print("Librerias importadas correctamente")
print(f"Fecha y hora: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
load_dotenv('../../.env')
USERNAME = os.getenv('IOL_USERNAME', 'tu_usuario_iol')
PASSWORD = os.getenv('IOL_PASSWORD', 'tu_password_iol')

if USERNAME == "tu_usuario_iol":
    print("ADVERTENCIA: Configura las credenciales en .env")
else:
    print(f"Credenciales configuradas - Usuario: {USERNAME}")

In [ ]:
try:
    client = IOLClient(USERNAME, PASSWORD)
    print("Cliente IOL creado correctamente")
except Exception as e:
    print(f"Error al crear cliente: {e}")
    client = None

## 1. Cotizaciones Masivas

**Tipos de instrumentos disponibles:**
- `acciones` - Acciones
- `cedears` - CEDEARs
- `opciones` - Opciones
- `titulosPublicos` - Bonos
- `obligacionesNegociables` - ONs
- `letras` - Letras

In [ ]:
# Cotizaciones masivas de acciones
if client:
    try:
        print("Obteniendo cotizaciones masivas de acciones argentinas...")
        cotizaciones = client.get_massive_quotes('acciones', 'argentina')
        
        if cotizaciones and cotizaciones.titulos:
            print(f"Total: {len(cotizaciones.titulos)} acciones")
            
            print("\nPrimeras 5 acciones:")
            for i, titulo in enumerate(cotizaciones.titulos[:5], 1):
                print(f"  {i}. {titulo.simbolo}: ${titulo.ultimo_precio} ({titulo.variacion_porcentual:+.2f}%)")
            
            # Buscar GGAL
            ggal = cotizaciones.get_by_symbol('GGAL')
            if ggal:
                print(f"\nGGAL: ${ggal.ultimo_precio} | Vol: {ggal.volumen:,}")
            
            # Top 3 por variacion
            print("\nTop 3 por variacion positiva:")
            for t in cotizaciones.sort_by_variation(ascending=False)[:3]:
                if t.variacion_porcentual > 0:
                    print(f"  {t.simbolo}: +{t.variacion_porcentual:.2f}%")
        else:
            print("No se obtuvieron cotizaciones")
    except Exception as e:
        print(f"Error: {e}")

In [ ]:
# Cotizaciones masivas de CEDEARs
if client:
    try:
        print("Obteniendo cotizaciones de CEDEARs...")
        cedears = client.get_massive_quotes('cedears', 'argentina')
        
        if cedears and cedears.titulos:
            print(f"Total: {len(cedears.titulos)} CEDEARs")
            
            # Buscar algunos conocidos
            for simbolo in ['AAPL', 'MSFT', 'GOOGL']:
                titulo = cedears.get_by_symbol(simbolo)
                if titulo:
                    print(f"  {simbolo}: ${titulo.ultimo_precio}")
    except Exception as e:
        print(f"Error: {e}")

## 2. Paneles de Cotizaciones

In [ ]:
# Panel MERVAL
if client:
    try:
        print("Obteniendo panel MERVAL...")
        merval = client.get_panel_quotes('acciones', 'merval', 'argentina')
        
        if merval and merval.titulos:
            print(f"Panel MERVAL: {len(merval.titulos)} acciones\n")
            
            for titulo in merval.titulos:
                icono = '+' if titulo.variacion_porcentual > 0 else '-' if titulo.variacion_porcentual < 0 else '='
                print(f"  {titulo.simbolo:6s} ${titulo.ultimo_precio:>10.2f} [{icono}{abs(titulo.variacion_porcentual):5.2f}%]")
            
            # Estadisticas
            variaciones = [t.variacion_porcentual for t in merval.titulos]
            alza = len([v for v in variaciones if v > 0])
            baja = len([v for v in variaciones if v < 0])
            print(f"\nEstadisticas: {alza} en alza, {baja} en baja")
            print(f"Variacion promedio: {sum(variaciones)/len(variaciones):+.2f}%")
    except Exception as e:
        print(f"Error: {e}")

## 3. Cotizacion Detallada

In [ ]:
# Cotizacion detallada con libro de ordenes
if client:
    try:
        print("Obteniendo cotizacion detallada de GGAL...")
        detalle = client.get_stock_quote_detailed('GGAL', 'bCBA')
        
        if detalle:
            print(f"\n{detalle.descripcion_titulo}")
            print(f"Precio: ${detalle.ultimo_precio}")
            print(f"Variacion: {detalle.variacion:+.2f}%")
            print(f"Apertura: ${detalle.apertura} | Max: ${detalle.maximo} | Min: ${detalle.minimo}")
            print(f"Volumen: {detalle.volumen_nominal:,}")
            
            # Libro de ordenes
            if detalle.puntas:
                print(f"\nLibro de ordenes ({len(detalle.puntas)} niveles):")
                for i, punta in enumerate(detalle.puntas[:5], 1):
                    print(f"  {i}. Compra: ${punta.precio_compra} ({punta.cantidad_compra}) | Venta: ${punta.precio_venta} ({punta.cantidad_venta})")
                
                # Spread
                if detalle.puntas[0]:
                    spread = detalle.puntas[0].precio_venta - detalle.puntas[0].precio_compra
                    print(f"\nSpread: ${spread:.2f}")
    except Exception as e:
        print(f"Error: {e}")

## Metodos RAW Disponibles

In [ ]:
print("METODOS RAW DE COTIZACIONES AVANZADAS")
print("="*50)
print("""
Para obtener respuestas en formato JSON crudo:

- client.get_massive_quotes_raw(instrumento, pais)
- client.get_panel_quotes_raw(instrumento, panel, pais)
- client.get_stock_quote_detailed_raw(simbolo, mercado)

Estos metodos retornan el JSON exacto de la API de IOL.
""")

## Limpieza

In [ ]:
if client:
    try:
        client.close()
        print("Cliente IOL cerrado correctamente")
    except Exception as e:
        print(f"Error al cerrar cliente: {e}")